# Supplementary Figure 5

In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(dplyr)
  library(ggplot2)
  library(patchwork)
  library(cowplot)
  library(colocboost)
})

source('../../2_main_figures/Figure4/data/colocboost_plot.R')
options(colocboost_plot.uncoloc_cos_label = 'Colocalization not supported')   # same legend wording as Figure 4c
gene_name <- 'ACE'

## Load cached figure data

In [ ]:
# Patch: data/Figure_S5b.rds was built by an external/untracked script using the same
glyco_final_samples <- colnames(fread('ftp_fgc_xqtl/gpQTL/unadjusted/analysis_ready/covariate_preprocessing/unadj_glyco.sorted.covariates_unadjusted_glyco.ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.pQTL.plink_qc.prune.pca.Marchenko_PC.gz'))[-1]
s5b_patch <- readRDS("data/Figure_S5b.rds")
n_before <- s5b_patch$plot_df %>% dplyr::filter(grepl("^gp_", pheno)) %>% dplyr::pull(sample) %>% n_distinct()
s5b_patch$plot_df <- s5b_patch$plot_df %>% dplyr::filter(!grepl("^gp_", pheno) | sample %in% glyco_final_samples)
n_after <- s5b_patch$plot_df %>% dplyr::filter(grepl("^gp_", pheno)) %>% dplyr::pull(sample) %>% n_distinct()
cat("Figure_S5b.rds glyco samples before/after 316-restriction:", n_before, "->", n_after, "\n")
saveRDS(s5b_patch, "data/Figure_S5b.rds")

s5a <- readRDS('data/Figure_S5a.rds')
main5a <- readRDS('../../2_main_figures/Figure5/data/Figure_5a.rds')
s5b <- readRDS('data/Figure_S5b.rds')

stopifnot(
  inherits(s5a$cb_res, 'colocboost'),
  length(main5a$new_id_vector) == 10L,
  all(c('start', 'end') %in% names(main5a$intron_df)),
  all(c('start', 'end') %in% names(main5a$ex_df)),
  all(c('pheno', 'genotype', 'pheno_value') %in% names(s5b$plot_df)),
  identical(names(s5b$geno_colors), c('G/G', 'G/A', 'A/A'))
)

## a. Colocalization locus plots

In [ ]:
glyph <- function(count, pch, fill, border) {
  list(count = count, pch = pch, fill = fill, border = border)
}

glycan_annotation <- function(site, hexnac, hex, fuc = 0, neuac = 0) {
  parts <- list(
    glyph(hexnac, 22, '#2F7ED8', '#1F4E79'),
    glyph(hex, 21, '#68C83E', '#2E6B25')
  )
  if (fuc > 0) parts <- c(parts, list(glyph(fuc, 24, '#D84A4A', '#7F1D1D')))
  if (neuac > 0) parts <- c(parts, list(glyph(neuac, 23, '#9667CF', '#5B3A8A')))
  list(
    line1 = paste('ACE', site), glycan = parts,
    # # Both fractions are relative to the panel's plotted x-span, so widening that range
    component_gap = 0.131, symbol_offset = 0.087
  )
}

outcome_annotations <- list(
  list(line1 = 'ACE', line2 = 'Protein'),
  glycan_annotation('N111', 2, 5),
  glycan_annotation('N111', 6, 4, 1, 1),
  glycan_annotation('N714', 2, 5),
  glycan_annotation('N714', 2, 6),
  glycan_annotation('N714', 2, 7),
  glycan_annotation('N509', 2, 9),
  glycan_annotation('N695', 2, 5),
  glycan_annotation('N695', 4, 4, 1),
  list(line1 = 'ADRD', line2 = 'GWAS')
)

outcome_order <- c(10L, seq_len(9L))
start_idx <- 4040L
end_idx <- 4240L

s5a_robust <- get_robust_colocalization(s5a$cb_res, cos_npc_cutoff = 0.2, npc_outcome_cutoff = 0.2, pvalue_cutoff = 0.0001)
lead_var_cos2 <- s5a_robust$cos_summary$top_variable[s5a_robust$cos_summary$focal_outcome == 'AD_Bellenguez'][1]
highlight_pos <- as.numeric(stringr::str_split(lead_var_cos2, ':', simplify = TRUE)[, 2])

library(tidyverse)
bed <- fread(file.path("data_output/colocalization/maf05/glycoQTL_maf05", "pQTL_glycoQTL_unadjusted_AD_pv1e4.bed"))
bed %>%
  filter(str_detect(event_ID, "AD_Bellenguez"),
         str_detect(event_ID, "ENSG00000159640")) %>%
  group_by(event_ID) %>%
  summarize(vars = str_c(variant_ID, collapse = ";\n"), vcps = str_c(vcp, collapse = ";\n"), .groups = "drop")

In [ ]:

# Restrict the highlight to the outcomes actually IN cos2
highlight_outcome_idx <- list(c(1, 3, 5, 6, 7, 8, 9, 10))

panel_y_limits <- vector('list', length(outcome_order))
panel_y_limits[[1]] <- c(0, 18)     # ADRD GWAS, max ~15.6
panel_y_limits[[2]] <- c(0, 19)     # ACE Protein, max ~16.2
panel_y_limits[[3]] <- c(0, 8)      # gp_2625, max ~6.7
panel_y_limits[[4]] <- c(0, 5.2)    # gp_2626, max ~4.2
panel_y_limits[[6]] <- c(0, 9.2)    # gp_2628, max ~7.7
panel_y_limits[[7]] <- c(0, 16)     # gp_2629, max ~13.6
panel_y_limits[[8]] <- c(0, 13.5)   # gp_2630, max ~11.2
panel_y_limits[[9]] <- c(0, 6.9)    # gp_2631, max ~5.7
panel_y_limits[[10]] <- c(0, 11.2)  # gp_2632, max ~9.4

draw_gene_track <- function(x_range, tick_at, column, axis_size) {
  plot.new()
  plot.window(xlim = x_range, ylim = c(-0.35, 0.35), xaxs = 'r', yaxs = 'i')
  with(main5a$intron_df, arrows(
    start, 0, end, 0, length = 0.045, angle = 25, code = 2, lwd = 0.7
  ))
  with(main5a$ex_df, rect(
    start, -0.18, end, 0.18, col = 'steelblue', border = NA
  ))
  axis(
    1, at = tick_at,
    labels = paste0(formatC(tick_at / 1e6, format = 'f', digits = 2), 'M'),
    cex.axis = axis_size
  )
  mtext('ACE', side = 1, line = 2.35, cex = 1.25, font = 2)
  box(lwd = 0.8)
}

draw_s5a <- function() {
  colocboost_plot(
    s5a_robust,
    grange = start_idx:end_idx, variant_coord = TRUE, xlab = '',
    outcome_idx = outcome_order, plot_cols = 5,
    outcome_annotations = outcome_annotations,
    bottom_track = draw_gene_track, compact_x_axes = TRUE, track_height = 0.37,
    add_highlight = TRUE, add_highlight_idx = highlight_pos,
    add_highlight_outcome_idx = highlight_outcome_idx,
    outcome_names = main5a$new_id_vector, add_highlight_style = 'coral_circle',
    panel_right_margin = 2.2, panel_inner_right_margin = 0.8,
    outcome_legend_size = 1.3,plot_focal_only = TRUE, 
    panel_y_limits = panel_y_limits,
    genomic_ticks = c(63500000, 64000000, 64200000),
    show_cos_to_uncoloc = TRUE,
    lab_style = c(1.75, 1),
    cos_color = c('#377EB8')
  )
}

f_s5a <- patchwork::wrap_elements(
  full = gridGraphics::echoGrob(draw_s5a, prefix = 'supplementary_figure_5a')
)

options(repr.plot.width = 22, repr.plot.height = 6.2)
f_s5a

## b. Genotype-stratified abundance

In [ ]:
panel_specs <- tibble::tibble(
  trait = c('pQTL', paste0('gp_', 2625:2632)),
  kind = c('protein', rep('glycan', 8)),
  site = c(NA, 'N111', 'N111', 'N714', 'N714', 'N714', 'N509', 'N695', 'N695'),
  counts = list(
    NULL, c(2, 5), c(6, 4, 1, 1), c(2, 5), c(2, 6),
    c(2, 7), c(2, 9), c(2, 5), c(4, 4, 1)
  )
)

plot_df <- as.data.frame(s5b$plot_df) %>%
  mutate(
    pheno = as.character(pheno),
    genotype = factor(as.character(genotype), levels = names(s5b$geno_colors))
  )
stopifnot(setequal(unique(plot_df$pheno), panel_specs$trait))

med_df <- plot_df %>%
  group_by(genotype, pheno) %>%
  summarise(median_y = median(pheno_value, na.rm = TRUE), .groups = 'drop')

make_panel_label_grob <- function(kind, site = NULL, counts = NULL) {
  is_glycan <- identical(kind, 'glycan')
  children <- list(
    grid::textGrob(
      if (is_glycan) paste('ACE', site) else 'ACE',
      x = grid::unit(0.97, 'npc'), y = grid::unit(0.96, 'npc'),
      just = c('right', 'top'),
      gp = grid::gpar(fontsize = 11.5, fontface = 'bold')
    ),
    grid::nullGrob()
  )

  if (!is_glycan) {
    children <- c(children, list(grid::textGrob(
      'Protein', x = grid::unit(0.97, 'npc'), y = grid::unit(0.90, 'npc'),
      just = c('right', 'top'), gp = grid::gpar(fontsize = 11)
    )))
  } else {
    pch <- c(22, 21, 24, 23)
    fill <- c('#2F7ED8', '#68C83E', '#D84A4A', '#9667CF')
    border <- c('#1F4E79', '#2E6B25', '#7F1D1D', '#5B3A8A')
    symbol_x <- 0.94 - rev(seq_along(counts) - 1) * 0.135
    for (ii in seq_along(counts)) {
      children <- c(children, list(
        grid::textGrob(
          paste0(counts[ii], '×'),
          x = grid::unit(symbol_x[ii] - 0.018, 'npc'),
          y = grid::unit(0.87, 'npc'),
          just = c('right', 'centre'), gp = grid::gpar(fontsize = 11.2)
        ),
        grid::pointsGrob(
          x = grid::unit(symbol_x[ii], 'npc'), y = grid::unit(0.87, 'npc'),
          pch = pch[ii], size = grid::unit(3.0, 'mm'),
          gp = grid::gpar(fill = fill[ii], col = border[ii], lwd = 0.8)
        )
      ))
    }
  }
  do.call(grid::grobTree, children)
}

make_genotype_panel <- function(trait, label_grob) {
  panel_df <- plot_df %>% filter(pheno == trait)
  panel_med <- med_df %>% filter(pheno == trait)
  panel_range <- range(panel_df$pheno_value, na.rm = TRUE)
  panel_span <- diff(panel_range)
  if (!is.finite(panel_span) || panel_span <= 0) panel_span <- 1
  panel_limits <- c(
    panel_range[1] - 0.05 * panel_span,
    panel_range[2] + 0.25 * panel_span
  )
  panel_breaks <- scales::breaks_pretty(n = 4)(panel_range)
  panel_breaks <- panel_breaks[
    panel_breaks >= panel_limits[1] & panel_breaks <= panel_range[2]
  ]

  ggplot(panel_df, aes(x = genotype, y = pheno_value)) +
    geom_boxplot(width = 0.5, alpha = 0.7, outlier.shape = NA, color = 'black') +
    geom_jitter(width = 0.2, size = 1.7, alpha = 0.6, aes(color = genotype)) +
    geom_line(
      data = panel_med,
      aes(x = as.numeric(genotype), y = median_y, group = 1),
      inherit.aes = FALSE, color = 'grey50', linewidth = 0.9, linetype = 'dashed'
    ) +
    scale_color_manual(values = s5b$geno_colors) +
    scale_y_continuous(limits = panel_limits, breaks = panel_breaks, expand = c(0, 0)) +
    annotation_custom(label_grob, xmin = -Inf, xmax = Inf, ymin = -Inf, ymax = Inf) +
    labs(x = NULL, y = NULL) +
    theme_bw(base_size = 13) +
    theme(
      axis.text.x = element_text(size = 12, face = 'bold'),
      axis.text.y = element_text(size = 11),
      legend.position = 'none',
      panel.grid = element_blank(),
      panel.border = element_rect(linewidth = 1.1),
      plot.margin = margin(t = 4, r = 7, b = 4, l = 7)
    )
}

s5b_panels <- lapply(seq_len(nrow(panel_specs)), function(ii) {
  spec <- panel_specs[ii, ]
  make_genotype_panel(
    spec$trait,
    make_panel_label_grob(spec$kind, spec$site, spec$counts[[1]])
  )
})

s5b_grid <- wrap_plots(s5b_panels, ncol = 5)
f_s5b <- cowplot::ggdraw() +
  cowplot::draw_plot(s5b_grid, x = 0.005, y = 0.07, width = 1, height = 0.93) +
  cowplot::draw_label(
    'Genotype (rs8077276)', x = 0.505, y = 0.050,
    hjust = 0.5, vjust = 0, fontface = 'bold', size = 15
  ) +
  cowplot::draw_label(
    'Normalized abundance', x = 0.008, y = 0.52, angle = 90,
    hjust = 0.5, vjust = 0, fontface = 'bold', size = 15
  )

options(repr.plot.width = 16, repr.plot.height = 6.5)
f_s5b

## Assemble and export

In [ ]:
combined_fig <- (free(f_s5a) / free(f_s5b)) +
  plot_layout(heights = c(1, 1.15)) +
  plot_annotation(tag_levels = 'a') &
  theme(
    plot.tag = element_text(size = 26, face = 'bold'),
    plot.tag.position = c(0.01, 0.99)
  )

# Width increased from 16 to 22in to match panel a's widened
options(repr.plot.width = 22, repr.plot.height = 13)
combined_fig

ggsave(
  'Supplementary_Figure5.pdf', combined_fig,
  device = grDevices::cairo_pdf, width = 22, height = 13,
  bg = 'white', limitsize = FALSE
)
cat('Saved: Supplementary_Figure5.pdf\n')